# Study 915 — K-1 vs 1099 — the teardown

PDBC (Form 1099, 1940-Act fund with a Cayman subsidiary) against DBC (Schedule K-1 commodity pool, Section 1256 futures), same manager, same index family. Three layers in increasing order of assumption: **(1)** the pre-tax race — pure tape; **(2)** the fee-gap reconciliation — one proxy; **(3)** the after-tax regime model — all assumption, fully swept.

Every real number below is frozen from `docs/results.md` (Fingerprint `0e33b87210d5`, as-of 2026-06-30). The live cells run synthetic data only and are labelled.

> 💡 **In plain words:** two funds, one commodity, different tax paperwork. Does the easy paperwork cost you anything?

In [1]:
R = {'start': '2014-11-07', 'end': '2026-06-30', 'n_days': 2926, 'fp': '0e33b87210d5', 'ric_sharpe': 0.151, 'ric_cagr': 3.01, 'ric_vol': 18.1, 'ric_dd': -49.3, 'ric_t': 0.51, 'k1_sharpe': 0.159, 'k1_cagr': 3.16, 'k1_vol': 17.9, 'k1_dd': -50.6, 'k1_t': 0.53, 'diff_ann': -0.1274, 't_diff': -0.21, 'te_ann': 5.3, 'corr': 0.9566, 'beta': 0.9626, 'r2': 0.9151, 'sharpe_adv': -0.0081, 'ci_lo': -1.0323, 'ci_hi': 0.7768, 'ci_pneg': 0.618, 'adv_ci_lo': -0.0587, 'adv_ci_hi': 0.0421, 'er_k1': 0.87, 'er_ric': 0.59, 'fee_gap_bp': 28, 'te_monthly': 1.46, 'n_months': 140, 'acf1': -0.43, 'diff_monthly': -0.1945, 't_monthly': -0.68, 'se_iid': 1.56, 'se_hac': 0.61, 'se_boot': 0.45, 'mde_iid': 3.11, 'mde_hac': 1.22, 'mde_boot': 0.91, 'era_e_n': 1547, 'era_e_diff': 0.008, 'era_e_t': 0.01, 'era_e_te': 7.04, 'era_l_n': 1377, 'era_l_diff': -0.319, 'era_l_t': -0.73, 'era_l_te': 2.0, 'n_years': 11, 'wins': 3, 'win_rate': 27, 'win_lo': 10, 'win_hi': 57, 'worst_year': 2025, 'worst_diff': -2.14, 'best_year': 2015, 'best_diff': 0.74, 'cost50_diff': -0.2136, 'cost50_t': -0.35, 'cost50_drag_bp': 8.62, 'gap_min': -0.36, 'gap_max': 0.588, 'gap_top_p0': 0.588, 'gap_top_p1': 0.219, 'gap_top_p2': -0.34, 'gap_mid_p1': 0.149, 'gap_low_p1': 0.021, 'regime_gap_low': 0.241, 'regime_gap_mid': 0.38, 'regime_gap_top': 0.438, 'pretax_cagr_k1': 3.328, 'pretax_cagr_ric': 3.055, 'aftertax_k1_mid': 2.034, 'aftertax_ric_mid': 2.183, 'usci_cagr': 4.68, 'uso_cagr': -6.7, 'bno_cagr': 1.85, 'dispersion_pp': 11.38, 'syn_planted': 4.0, 'syn_recovered': 4.95, 'syn_t': -4.03, 'syn_null_mean': 0.453, 'syn_null_sd': 1.057, 'syn_null_fire': 0}

## Layer 1 — the pre-tax race

Both legs buy-and-hold, both measured **excess of cash** (BIL), so the collateral T-bill yield — ~5%/yr in 2023-2026 and a large fraction of a commodity fund's return in those years — is netted out of *both* sides. **One execution lag**: the wrapper is bought at the first common close and the first return counted is the next session's. **No shorting**, hence no borrow anywhere in this study.

In [2]:
print(f"DBC  (K-1)  : exSharpe {R['k1_sharpe']:+.3f}  CAGR {R['k1_cagr']:+.2f}%  "
      f"vol {R['k1_vol']:.1f}%  MaxDD {R['k1_dd']:.1f}%  HAC t {R['k1_t']:+.2f}")
print(f"PDBC (1099) : exSharpe {R['ric_sharpe']:+.3f}  CAGR {R['ric_cagr']:+.2f}%  "
      f"vol {R['ric_vol']:.1f}%  MaxDD {R['ric_dd']:.1f}%  HAC t {R['ric_t']:+.2f}")
print()
print(f"difference (1099 - K-1) : {R['diff_ann']:+.4f}%/yr   HAC t = {R['t_diff']:+.2f}")
print(f"  paired block-bootstrap 95% CI : [{R['ci_lo']:+.4f}%, {R['ci_hi']:+.4f}%]  "
      f"P(<0) = {R['ci_pneg']:.3f}")
print(f"excess-of-cash Sharpe advantage : {R['sharpe_adv']:+.4f}  "
      f"CI [{R['adv_ci_lo']:+.4f}, {R['adv_ci_hi']:+.4f}]")
print(f"tracking: corr {R['corr']:.4f}  beta {R['beta']:.4f}  R2 {R['r2']:.4f}")

DBC  (K-1)  : exSharpe +0.159  CAGR +3.16%  vol 17.9%  MaxDD -50.6%  HAC t +0.53
PDBC (1099) : exSharpe +0.151  CAGR +3.01%  vol 18.1%  MaxDD -49.3%  HAC t +0.51

difference (1099 - K-1) : -0.1274%/yr   HAC t = -0.21
  paired block-bootstrap 95% CI : [-1.0323%, +0.7768%]  P(<0) = 0.618
excess-of-cash Sharpe advantage : -0.0081  CI [-0.0587, +0.0421]
tracking: corr 0.9566  beta 0.9626  R2 0.9151


## Layer 2 — what the fee gap predicted vs what the tape delivered

**PROXY / ASSUMPTION carrying hindsight:** the two **current** prospectus expense ratios, applied to the whole 2014-2026 window. They are *not* measured here and are never subtracted from a return series — both tapes are already net of the real, time-varying fees. They are quoted only to check the sign of the prediction, so the hindsight never touches a backtest.

In [3]:
print(f"expense ratio (PROXY): DBC {R['er_k1']:.2f}%  PDBC {R['er_ric']:.2f}%")
print(f"  -> the fee gap predicts the 1099 wrapper WINS by {R['fee_gap_bp']} bp/yr")
print(f"  -> the tape says it LOST by {abs(R['diff_ann'])*100:.0f} bp/yr")
print(f"  -> a {R['fee_gap_bp'] + abs(R['diff_ann'])*100:.0f} bp/yr shortfall, "
      f"inside one standard error ({R['se_hac']:.2f}%/yr)")
print('  -> i.e. the tape cannot tell the fee prediction apart from zero either;')
print('     active/basket drift is a candidate, but it is NOT measured here.')

expense ratio (PROXY): DBC 0.87%  PDBC 0.59%
  -> the fee gap predicts the 1099 wrapper WINS by 28 bp/yr
  -> the tape says it LOST by 13 bp/yr
  -> a 41 bp/yr shortfall, inside one standard error (0.61%/yr)
  -> i.e. the tape cannot tell the fee prediction apart from zero either;
     active/basket drift is a candidate, but it is NOT measured here.


## The nuisance term — daily TE is mostly microstructure

The daily difference carries a lag-1 autocorrelation of −0.43: the two closes disagree and re-converge. Naive √252 scaling therefore inflates the tracking error threefold relative to the divergence a holder lives through.

> 💡 **In plain words:** the two prices bounce against each other day to day, then snap back. Look monthly and the wobble mostly cancels.

In [4]:
print(f"daily-scaled TE {R['te_ann']:.2f}%/yr   vs   monthly TE {R['te_monthly']:.2f}%/yr "
      f"({R['n_months']} months)")
print(f"lag-1 autocorrelation of the daily difference: {R['acf1']:+.3f}")
print(f"monthly difference {R['diff_monthly']:+.4f}%/yr (HAC t = {R['t_monthly']:+.2f}) "
      f"-> same null, cleaner measurement")

daily-scaled TE 5.30%/yr   vs   monthly TE 1.46%/yr (140 months)
lag-1 autocorrelation of the daily difference: -0.430
monthly difference -0.1945%/yr (HAC t = -0.68) -> same null, cleaner measurement


## Power — is this null informative or merely blind?

Three standard errors for the same quantity. The i.i.d. one is wrong (negative autocorrelation); the HAC and bootstrap ones agree.

In [5]:
for tag, se, mde in [('i.i.d.', R['se_iid'], R['mde_iid']),
                     ('HAC (Newey-West)', R['se_hac'], R['mde_hac']),
                     ('block bootstrap', R['se_boot'], R['mde_boot'])]:
    print(f"{tag:18s}: SE {se:.2f}%/yr   minimum detectable at |t|=2: {mde:.2f}%/yr")
print()
print('=> the claim is "no wrapper cost above ~0.9-1.2%/yr", not "no wrapper cost".')

i.i.d.            : SE 1.56%/yr   minimum detectable at |t|=2: 3.11%/yr
HAC (Newey-West)  : SE 0.61%/yr   minimum detectable at |t|=2: 1.22%/yr
block bootstrap   : SE 0.45%/yr   minimum detectable at |t|=2: 0.91%/yr

=> the claim is "no wrapper cost above ~0.9-1.2%/yr", not "no wrapper cost".


## Era cut — and a converging pair

Null in both halves. Note the tracking error *falling* as PDBC grew out of its illiquid first years: the wrappers converged rather than diverged. (Both TEs are the naive √252-scaled daily figure the cell above shows to be inflated — the *fall* is the point, not the level.)

In [6]:
print(f"2014-11..2020-12 (n={R['era_e_n']}): diff {R['era_e_diff']:+.3f}%/yr "
      f"(t={R['era_e_t']:+.2f})  TE {R['era_e_te']:.2f}%")
print(f"2021-01..2026-06 (n={R['era_l_n']}): diff {R['era_l_diff']:+.3f}%/yr "
      f"(t={R['era_l_t']:+.2f})  TE {R['era_l_te']:.2f}%   <- TE more than halved")

2014-11..2020-12 (n=1547): diff +0.008%/yr (t=+0.01)  TE 7.04%
2021-01..2026-06 (n=1377): diff -0.319%/yr (t=-0.73)  TE 2.00%   <- TE more than halved


## Calendar years, sign test, and the cost sweep

Costs are amortised over the holding period — buy-and-hold means one round trip, so 2 x spread / 11.6 years. The sweep is deliberately asymmetric: the extra spread is charged to the 1099 leg only.

In [7]:
print(f"1099 wins {R['wins']}/{R['n_years']} complete years "
      f"({R['win_rate']}%, Wilson 95% CI [{R['win_lo']}%, {R['win_hi']}%]) "
      f"-> a coin flip is inside the interval")
print(f"worst year {R['worst_year']}: {R['worst_diff']:+.2f} pp   "
      f"best year {R['best_year']}: {R['best_diff']:+.2f} pp   "
      f"(sign reverses -> not a structural wrapper cost)")
print('   PDBC is actively managed, so basket drift is the candidate --')
print('   INTERPRETATION, not a measurement: no holdings file was parsed.')
print()
print(f"+50 bp one-way charged to the 1099 leg -> {R['cost50_drag_bp']:.2f} bp/yr drag, "
      f"difference {R['cost50_diff']:+.4f}%/yr (t={R['cost50_t']:+.2f})")
print('=> at an 11.6-year horizon, trading friction is not the story.')

1099 wins 3/11 complete years (27%, Wilson 95% CI [10%, 57%]) -> a coin flip is inside the interval
worst year 2025: -2.14 pp   best year 2015: +0.74 pp   (sign reverses -> not a structural wrapper cost)
   PDBC is actively managed, so basket drift is the candidate --
   INTERPRETATION, not a measurement: no holdings file was parsed.

+50 bp one-way charged to the 1099 leg -> 8.62 bp/yr drag, difference -0.2136%/yr (t=-0.35)
=> at an 11.6-year horizon, trading friction is not the story.


## Layer 3 — the after-tax model. Every input is an ASSUMPTION

**K-1 leg:** §1256 contracts, marked to market each 31 December, taxed at the 60/40 blend; collateral interest (proxied by BIL's total return on the opening balance) taxed as ordinary income; net 1256 losses carried forward; tax paid out of the account; annual MTM steps the basis up so **nothing is owed at liquidation**.

**1099 leg:** ordinary distributions = `payout_share` × the same interest proxy, taxed annually and reinvested (basis rises, the tax withdrawal removes basis pro rata); the remainder **deferred** and taxed once at the long-term rate on sale.

**Not modelled:** state tax, the accountant time a K-1 costs, filing extensions, UBTI inside a retirement account, and PDBC's IRA eligibility. No filing was parsed — this is a transparent model, not a measurement.

> 💡 **In plain words:** the K-1 fund gets a better tax rate but pays every year; the 1099 fund gets a worse rate on part of the return but delays the rest. Which wins is arithmetic — and it depends on how much the 1099 fund hands out.

In [8]:
print('after-tax gap (1099 - K-1), pp/yr, across the assumption grid')
print(f"  top bracket 40.8/23.8: payout 0.0 -> {R['gap_top_p0']:+.3f}   "
      f"payout 1.0 -> {R['gap_top_p1']:+.3f}   payout 2.0 -> {R['gap_top_p2']:+.3f}")
print(f"  32%+NIIT   35.8/18.8: payout 1.0 -> {R['gap_mid_p1']:+.3f}")
print(f"  24%/15%             : payout 1.0 -> {R['gap_low_p1']:+.3f}")
print()
print(f"full-grid range: [{R['gap_min']:+.3f}, {R['gap_max']:+.3f}] pp/yr -- THE SIGN FLIPS.")
print('=> the after-tax winner is chosen by the payout assumption, not by the tape.')

after-tax gap (1099 - K-1), pp/yr, across the assumption grid
  top bracket 40.8/23.8: payout 0.0 -> +0.588   payout 1.0 -> +0.219   payout 2.0 -> -0.340
  32%+NIIT   35.8/18.8: payout 1.0 -> +0.149
  24%/15%             : payout 1.0 -> +0.021

full-grid range: [-0.360, +0.588] pp/yr -- THE SIGN FLIPS.
=> the after-tax winner is chosen by the payout assumption, not by the tape.


## The regime alone — both tax rules on the SAME return stream

Applying both regimes to DBC's own annual returns removes the tracking difference, leaving only the value of deferral against the 60/40 rate.

In [9]:
print(f"22/15      : regime gap {R['regime_gap_low']:+.3f} pp/yr to the 1099 wrapper")
print(f"35.8/18.8  : regime gap {R['regime_gap_mid']:+.3f} pp/yr")
print(f"40.8/23.8  : regime gap {R['regime_gap_top']:+.3f} pp/yr")
print()
print(f"pre-tax CAGR (complete years): K-1 {R['pretax_cagr_k1']:+.3f}%  "
      f"1099 {R['pretax_cagr_ric']:+.3f}%")
print(f"after tax at 35.8/18.8, payout 1.0: K-1 {R['aftertax_k1_mid']:+.3f}%  "
      f"1099 {R['aftertax_ric_mid']:+.3f}%")
print('=> deferral is worth 24-44 bp/yr; the tracking difference takes back ~27 bp/yr.')

22/15      : regime gap +0.241 pp/yr to the 1099 wrapper
35.8/18.8  : regime gap +0.380 pp/yr
40.8/23.8  : regime gap +0.438 pp/yr

pre-tax CAGR (complete years): K-1 +3.328%  1099 +3.055%
after tax at 35.8/18.8, payout 1.0: K-1 +2.034%  1099 +2.183%
=> deferral is worth 24-44 bp/yr; the tracking difference takes back ~27 bp/yr.


## Context — the size of the decision next door

Survivorship note: no dead commodity ETP was dropped, and both wrappers are the live vehicles an investor actually faced from PDBC's inception. The *pair* was chosen because it is the famous K-1/No-K-1 twin — selection on salience, not on performance.

In [10]:
print(f"USCI (K-1)  CAGR {R['usci_cagr']:+.2f}%")
print(f"DBC  (K-1)  CAGR {R['k1_cagr']:+.2f}%")
print(f"PDBC (1099) CAGR {R['ric_cagr']:+.2f}%")
print(f"BNO  (K-1)  CAGR {R['bno_cagr']:+.2f}%")
print(f"USO  (K-1)  CAGR {R['uso_cagr']:+.2f}%")
print()
print(f"index/vehicle dispersion {R['dispersion_pp']:.2f} pp/yr  vs  "
      f"wrapper difference {abs(R['diff_ann']):.2f} pp/yr  (~90x)")

USCI (K-1)  CAGR +4.68%
DBC  (K-1)  CAGR +3.16%
PDBC (1099) CAGR +3.01%
BNO  (K-1)  CAGR +1.85%
USO  (K-1)  CAGR -6.70%

index/vehicle dispersion 11.38 pp/yr  vs  wrapper difference 0.13 pp/yr  (~90x)


## Synthetic control — SYNTHETIC DATA, never the real tape

Two wrappers on a shared commodity factor plus independent tracking noise. Plant a known drag on one: the race must recover it at |*t*| ≥ 2. Switch the drag off: the race must stay quiet across a panel of independent seeds. This proves the null above is a property of the wrappers, not of a sleepy estimator.

In [11]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from wrapper_tax import data, strategy as st

planted, truth = data.synthetic_daily(signal_strength=1.0, seed=915)
det = st.synthetic_detect(planted, n_boot=600)
print('SYNTHETIC planted drag %.2f%%/yr -> recovered %.2f%%/yr  (HAC t %+.2f, CI [%+.2f%%, %+.2f%%])'
      % (truth['planted_drag_ann']*100, det['estimated_drag_ann']*100,
         det['t_diff'], det['ci_low']*100, det['ci_high']*100))

panel = data.synthetic_panel(n_pairs=8, signal_strength=0.0, seed=915)
diffs = np.array([st.race(p, 'ric', 'k1', 'cash')['diff_ann'] for p, _ in panel])
ts = np.array([st.race(p, 'ric', 'k1', 'cash')['t_diff'] for p, _ in panel])
print('SYNTHETIC null x8: mean drift %+.3f%%/yr (sd %.3f%%), |t|>=2 in %d/8'
      % (diffs.mean()*100, diffs.std(ddof=1)*100, int((np.abs(ts) >= 2).sum())))

SYNTHETIC planted drag 4.00%/yr -> recovered 4.95%/yr  (HAC t -4.03, CI [-7.42%, -2.74%])


SYNTHETIC null x8: mean drift +0.453%/yr (sd 1.057%), |t|>=2 in 0/8


## Verdict

- **Signal — None.** The pre-tax difference is **-0.1274%/yr** at **HAC *t* = -0.21**, paired bootstrap CI [-1.03%, +0.78%]; the excess-of-cash Sharpe advantage is -0.0081, CI [-0.059, +0.042]. Null in both eras (|*t*| ≤ 0.73), 3/11 annual wins with a coin flip inside the Wilson interval, and a 50 bp punitive spread moves the answer 9 bp. The sample had the power to detect anything above ~0.9-1.2%/yr, so this is an informative null, not a blind one. The after-tax layer cannot promote it: the modelled gap spans [-0.36, +0.59] pp/yr and **changes sign** inside the assumption grid. The synthetic control recovers a planted 4.0%/yr drag (4.95%/yr, *t* = -4.03) and stays quiet on the null (0/8), so the estimator works.
- **Tradability — Mirage.** Nothing to bank. The wrapper difference is tens of basis points against 1.46-5.30%/yr of tracking error and 11.4 pp/yr of index-choice dispersion, and the after-tax gap is smaller than the assumptions that generate it. The usable conclusion is the non-finding — the No-K-1 convenience is, on this evidence, **free** — and free convenience is not alpha.